# Exercise 3

This notebook now follows Problem 3 of the exam brief: choose a neural quantum state architecture, train it variationally, benchmark the small-system training path against exact diagonalization, and then study larger-system behavior and entanglement trends after training. Bonus GHZ material is kept explicitly separate at the end.


In [ ]:
from notebook_bootstrap import bootstrap_notebook

bootstrap_notebook(enable_x64=True)

In [ ]:
import pandas as pd

from exercise_report_helper import (
    build_output_manifest,
    ensure_report_output_dir,
    plot_energy_benchmark,
    plot_entropy_scan,
    plot_training_history,
    save_report_figure,
    save_report_table,
)
from nqs.workflows import run_ghz_bonus_workflow, run_hamiltonian_system_size_sweep, run_vmc_experiment, tfim_proxy_sweep_points

In [ ]:
selected_model = {'model_name': 'RBM', 'model_kwargs': {'alpha': 2}}

small_training_config = {
    'n_iter': 40,
    'n_samples': 128,
    'n_discard_per_chain': 32,
    'n_chains': 16,
    'callback_every': 2,
}

large_training_config = {
    'n_iter': 30,
    'n_samples': 256,
    'n_discard_per_chain': 32,
    'n_chains': 16,
    'callback_every': 2,
}

exercise_output_dir = ensure_report_output_dir('exercise_3')
exercise_output_dir

## 3/a Architecture Choice

Based on the random-state study from Exercise 2, we choose the `RBM` as the training ansatz for the main VMC path. This is the smallest bounded choice that still fits the shared model/sampler/variational-state interface, has a compact parameterization, and gives a clean reference point before comparing more expensive architectures.

The choice is therefore pragmatic rather than universal: for this exam pass, `RBM(alpha=2)` gives a stable training target while keeping the notebook readable and the training costs modest.


In [ ]:
pd.DataFrame([{
    'selected_model': selected_model['model_name'],
    'model_kwargs': selected_model['model_kwargs'],
    'justification': 'compact baseline with stable shared workflow',
}])

## 3/b VMC Setup For The TFIM Ground State Search

We train on the 1D TFIM realized through the current `(L, 1)` lattice backend with open boundaries. The VMC driver minimizes the energy while the entropy callback logs the Renyi-2 entropy during training. This keeps the experiment close to the exam brief without introducing a new graph subsystem just for the notebook.


## 3/c Small-System Benchmarks: Energy And Renyi-2 During Training

We first benchmark the training loop on a small TFIM chain where exact diagonalization is still cheap. To answer the exam question about parameter dependence, we compare the critical point `g=1.0` against an off-critical paramagnetic point `g=1.5` on the same `4x1` chain.


In [ ]:
exercise_3_small_critical = run_vmc_experiment(
    **selected_model,
    lattice_shape=(4, 1),
    pbc=False,
    hamiltonian='tfim',
    J=1.0,
    h=1.0,
    entropy_n_independent_runs=8,
    **small_training_config,
)

exercise_3_small_paramagnetic = run_vmc_experiment(
    **selected_model,
    lattice_shape=(4, 1),
    pbc=False,
    hamiltonian='tfim',
    J=1.0,
    h=1.5,
    entropy_n_independent_runs=8,
    **small_training_config,
)

exercise_3_small_summary = pd.DataFrame([
    {
        'sweep_label': 'critical_g1.0',
        'exact_ground_energy': exercise_3_small_critical['exact']['ground_energy'],
        'final_energy': exercise_3_small_critical['final_energy'],
        'energy_error': exercise_3_small_critical['energy_error'],
        'final_renyi2_entropy': exercise_3_small_critical['final_entropy'],
    },
    {
        'sweep_label': 'paramagnetic_g1.5',
        'exact_ground_energy': exercise_3_small_paramagnetic['exact']['ground_energy'],
        'final_energy': exercise_3_small_paramagnetic['final_energy'],
        'energy_error': exercise_3_small_paramagnetic['energy_error'],
        'final_renyi2_entropy': exercise_3_small_paramagnetic['final_entropy'],
    },
])

exercise_3_small_history = pd.concat([
    exercise_3_small_critical['history_df'].assign(sweep_label='critical_g1.0'),
    exercise_3_small_paramagnetic['history_df'].assign(sweep_label='paramagnetic_g1.5'),
], ignore_index=True)

exercise_3_small_summary

In [ ]:
small_energy_figure = plot_training_history(
    exercise_3_small_history,
    'energy',
    line_column='sweep_label',
    title='Exercise 3 small-system energy during training',
)
small_energy_figure

In [ ]:
small_entropy_figure = plot_training_history(
    exercise_3_small_history,
    'renyi2_entropy',
    line_column='sweep_label',
    title='Exercise 3 small-system Renyi-2 during training',
)
small_entropy_figure

In [ ]:
small_benchmark_figure = plot_energy_benchmark(
    exercise_3_small_summary,
    title='Exercise 3 exact-vs-VMC small-system benchmark',
)
small_benchmark_figure

At this stage the energy benchmark and the entropy history answer the key Problem 3 question: the training dynamics depend on the Hamiltonian parameter. The critical point typically shows slower convergence and stronger entanglement than the off-critical paramagnetic point, so energy and Renyi-2 do not settle in the same way across the two runs.


## 3/d Larger-System Training Without Relying On ED

We now repeat the training workflow on larger TFIM chains. The point here is not to produce a perfect benchmark, but to see how the training history and the final subsystem-size dependence of Renyi-2 change as the system grows and as we compare the critical point against the off-critical paramagnetic regime.


In [ ]:
exercise_3_large_points = [
    {
        **point,
        'label': f"critical_g1.0_L{point['lattice_shape'][0]}x1",
    }
    for point in tfim_proxy_sweep_points([6, 8], h=1.0, pbc=False)
] + [
    {
        **point,
        'label': f"paramagnetic_g1.5_L{point['lattice_shape'][0]}x1",
    }
    for point in tfim_proxy_sweep_points([6, 8], h=1.5, pbc=False)
]

exercise_3_large = run_hamiltonian_system_size_sweep(
    sweep_points=exercise_3_large_points,
    entropy_n_independent_runs=8,
    **selected_model,
    **large_training_config,
)

exercise_3_large['summary_table']

In [ ]:
large_energy_history_figure = plot_training_history(
    exercise_3_large['training_history_table'],
    'energy',
    line_column='sweep_label',
    title='Exercise 3 larger-system energy during training',
)
large_energy_history_figure

In [ ]:
large_entropy_history_figure = plot_training_history(
    exercise_3_large['training_history_table'],
    'renyi2_entropy',
    line_column='sweep_label',
    title='Exercise 3 larger-system Renyi-2 during training',
)
large_entropy_history_figure

In [ ]:
exercise_3_large_entropy_scan = pd.concat([
    sweep_result['entropy_scan_table'].assign(sweep_label=sweep_result['label'])
    for sweep_result in exercise_3_large['sweep_results']
], ignore_index=True)

large_entropy_scan_figure = plot_entropy_scan(
    exercise_3_large_entropy_scan.rename(columns={'renyi2': 'renyi2'}),
    line_column='sweep_label',
    title='Exercise 3 final Renyi-2 vs subsystem size after training',
)
large_entropy_scan_figure

## 3/e Bonus: GHZ Training Check

The GHZ task is kept explicitly as a bonus. It is a different training target from the TFIM ground-state search and is therefore separated from the required benchmark path above.


In [ ]:
ghz_config = {
    'lattice_shape': (2, 2),
    'n_iter': 20,
    'n_samples': 128,
    'n_discard_per_chain': 16,
    'n_chains': 16,
    'callback_every': 2,
}

exercise_3_ghz = run_ghz_bonus_workflow(
    **selected_model,
    **ghz_config,
)

exercise_3_ghz_history = exercise_3_ghz['history_df'].copy()
exercise_3_ghz_history['model'] = 'RBM_GHZ'

pd.DataFrame([exercise_3_ghz['ghz_metrics']])

In [ ]:
ghz_history_figure = plot_training_history(
    exercise_3_ghz_history,
    'energy',
    line_column='model',
    title='Exercise 3 GHZ bonus energy during training',
)
ghz_history_figure

## 3/f Bonus Follow-Up Note

The entanglement-spectrum-tail question is left as a follow-up rather than forced into this notebook. A convincing answer would require a more deliberate comparison against a truncation-based reference such as an MPS workflow or a dedicated sample-number convergence study focused on the smallest Schmidt values.


## Export Exercise 3 Artifacts

Persist the required training tables and the main benchmark figures for later report assembly.

In [ ]:
small_summary_paths = save_report_table(exercise_3_small_summary, 'exercise_3_small_summary', output_dir=exercise_output_dir)
small_history_paths = save_report_table(exercise_3_small_history, 'exercise_3_small_history', output_dir=exercise_output_dir)
large_summary_paths = save_report_table(exercise_3_large['summary_table'], 'exercise_3_large_summary', output_dir=exercise_output_dir)
large_history_paths = save_report_table(exercise_3_large['training_history_table'], 'exercise_3_large_history', output_dir=exercise_output_dir)
large_entropy_scan_paths = save_report_table(exercise_3_large_entropy_scan, 'exercise_3_large_entropy_scan', output_dir=exercise_output_dir)
ghz_paths = save_report_table(pd.DataFrame([exercise_3_ghz['ghz_metrics']]), 'exercise_3_ghz_summary', output_dir=exercise_output_dir)
small_energy_path = save_report_figure(small_energy_figure, 'exercise_3_small_energy', output_dir=exercise_output_dir)
small_entropy_path = save_report_figure(small_entropy_figure, 'exercise_3_small_entropy', output_dir=exercise_output_dir)
small_benchmark_path = save_report_figure(small_benchmark_figure, 'exercise_3_small_benchmark', output_dir=exercise_output_dir)
large_energy_path = save_report_figure(large_energy_history_figure, 'exercise_3_large_energy', output_dir=exercise_output_dir)
large_entropy_path = save_report_figure(large_entropy_history_figure, 'exercise_3_large_entropy', output_dir=exercise_output_dir)
large_entropy_scan_path = save_report_figure(large_entropy_scan_figure, 'exercise_3_large_entropy_scan', output_dir=exercise_output_dir)
ghz_history_path = save_report_figure(ghz_history_figure, 'exercise_3_ghz_history', output_dir=exercise_output_dir)

build_output_manifest([
    {'section': 'exercise_3', 'name': 'small_summary', 'path': str(small_summary_paths['csv'])},
    {'section': 'exercise_3', 'name': 'small_history', 'path': str(small_history_paths['csv'])},
    {'section': 'exercise_3', 'name': 'large_summary', 'path': str(large_summary_paths['csv'])},
    {'section': 'exercise_3', 'name': 'large_history', 'path': str(large_history_paths['csv'])},
    {'section': 'exercise_3', 'name': 'large_entropy_scan', 'path': str(large_entropy_scan_paths['csv'])},
    {'section': 'exercise_3', 'name': 'ghz_summary', 'path': str(ghz_paths['csv'])},
    {'section': 'exercise_3', 'name': 'small_energy_figure', 'path': str(small_energy_path)},
    {'section': 'exercise_3', 'name': 'small_entropy_figure', 'path': str(small_entropy_path)},
    {'section': 'exercise_3', 'name': 'small_benchmark_figure', 'path': str(small_benchmark_path)},
    {'section': 'exercise_3', 'name': 'large_energy_figure', 'path': str(large_energy_path)},
    {'section': 'exercise_3', 'name': 'large_entropy_figure', 'path': str(large_entropy_path)},
    {'section': 'exercise_3', 'name': 'large_entropy_scan_figure', 'path': str(large_entropy_scan_path)},
    {'section': 'exercise_3', 'name': 'ghz_history_figure', 'path': str(ghz_history_path)},
])